<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-05-bigquery-ml/lesson-5.5-retrieval-features/notebooks/GCP_Capstone_5.5_Retrieval_Features.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5.5 Engineer Retrieval Features
**Netsetos GenAI Engineering — GCP Capstone** · Module 5 · new in v1.1

Four modules of retrieval have treated a chunk as text plus a vector. It is also a **row of facts**: how long, how fresh, what kind of document, whether it carries anything you must not index.

This notebook builds those facts and puts them to work:

- a **`chunk_metadata`** feature table and an append-only **`ingest_events`** log
- **cheap features** in pure SQL, and one **expensive feature** via `AI.GENERATE_TABLE`
- a **PII gate** that keeps chunks out of the index rather than ranking them down
- Vector Search **`restricts`** and **`numeric_restricts`** — filtering at the index, not after it
- a **sparse vector** beside the dense one in the same index
- a **BQML `TRANSFORM`** routing classifier whose preprocessing travels with the model
- a **data-quality gate** that blocks the index rebuild

Prerequisites: lesson **5.3**, whose fixture cell seeds `rag_data.doc_chunks` with twelve ACME policy clauses, and lessons **5.1–5.4** for the dataset and the BigQuery↔Vertex connection. Cell 1 checks for the table and says so if it is missing. *Facts verified 2026-09-03.*

## Setup
**One** client, and the reason is the interesting part. The rule from lesson 1.1 still holds: Gemini 3.x generation is served **only** from `global`, embeddings are **regional-only**. But this lesson's generation does not happen in Python at all - Cells 4 and 11 generate *inside BigQuery*, through a remote model. So the only Python client here is the regional one that embeds, in Cell 8.

`EMBED_DIMS = 768` is not optional. `gemini-embedding-001` returns **3072** dimensions unless you ask for 768, and an unpinned call silently produces vectors that cannot be compared with the rest of the course (lesson 2.4).

In [ ]:
!pip install -q google-genai==2.21.0 google-cloud-bigquery google-cloud-aiplatform \
                 rank-bm25==0.2.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"      # CHANGE THIS
DATASET    = "rag_data"                 # where lessons 5.1-5.4 already work; used everywhere
TENANT     = "acme"

from google.cloud import bigquery
from google import genai
from google.genai import types

bq = bigquery.Client(project=PROJECT_ID)

# One client, and the reason is the interesting part. The rule from lesson 1.1 still holds -
# Gemini 3.x generation is served ONLY from the global endpoint, embeddings are regional-only - but
# this lesson's generation does not happen in Python at all. Steps 4 and 11 generate INSIDE
# BigQuery, through a remote model that reaches Vertex AI over a connection. So the only Python
# client here is the regional one that embeds, in Step 8.
emb_client = genai.Client(enterprise=True, project=PROJECT_ID, location="us-central1")

CLASSIFY_MODEL = "gemini-3.1-flash-lite"   # bulk classification: cheapest per token
EMBED_MODEL    = "gemini-embedding-001"    # the corpus model since 2.4 - pinned to 768 below
EMBED_DIMS     = 768                       # this model returns 3072 unless you ask for 768

def run(sql: str):
    """Run a query and return the rows as a list. Prints the bytes billed, because SQL costs money."""
    job = bq.query(sql)
    rows = list(job.result())
    print(f"  {job.total_bytes_billed or 0:,} bytes billed | {len(rows)} rows")
    return rows

print("ready:", PROJECT_ID, "|", DATASET)

## Cell 1: Two tables — what is true now, and what changed
`chunk_metadata` is one row per chunk, updated in place: *what do we currently believe about this chunk?* `ingest_events` is append-only: *when did that change, and why?*

> An overwritten row has no memory. The auditor's question is never “what is indexed?” but “why was this chunk indexed on the fourteenth?” The `withheld` event in Cell 5 is the row you will be asked to produce.

`PARTITION BY DATE(featured_at)` partitions on *when we last featured the chunk*, which makes "what did the pipeline touch on the fourteenth" cheap and gives you a retention lever. Note what it does **not** do: the merge stamps every updated row with the current timestamp, so updated rows migrate into today's partition. This is an ingest-date partition, not a stable one. Partition on the document's revision date if you need partitions that hold still.

`CLUSTER BY tenant_id, doc_type` earns its keep on a multi-tenant corpus. On this single-tenant fixture the leading column is constant, so it buys nothing here and everything in production.

In [ ]:
# chunk_metadata is ONE ROW PER CHUNK, keyed the same way Firestore keys chunks (4.1).
# It holds no text and no vectors: it is the feature table, and the join key back to the corpus.
DDL = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET}.chunk_metadata` (
  chunk_id      STRING  NOT NULL OPTIONS(description = 'Firestore document id in chunks'),
  tenant_id     STRING  NOT NULL,
  source_uri   STRING,
  page_start    INT64,
  page_end      INT64,

  -- cheap features: computed in SQL, no model call (Step 3)
  token_count   INT64   OPTIONS(description = 'Approximate: characters / 4'),
  heading_depth INT64   OPTIONS(description = '0 = document root, 3 = deeply nested subsection'),
  has_table     BOOL,
  has_figure    BOOL,
  language      STRING  OPTIONS(description = 'BCP-47-ish: en, hi, mixed'),
  freshness_days INT64  OPTIONS(description = 'Days since the source document was last revised'),

  -- expensive features: one model call per chunk, run nightly (Steps 4 and 5)
  doc_type      STRING  OPTIONS(description = 'policy | contract | report | form | correspondence'),
  doc_type_conf FLOAT64,
  pii_flag      BOOL    OPTIONS(description = 'TRUE keeps the chunk OUT of the shared index'),

  -- provenance
  featured_at   TIMESTAMP OPTIONS(description = 'When this row was last recomputed')
)
PARTITION BY DATE(featured_at)
CLUSTER BY tenant_id, doc_type
"""
run(DDL)

# ingest_events is the append-only log: one row per chunk per ingest, never updated.
# chunk_metadata answers "what is true now"; ingest_events answers "what changed, and when".
run(f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET}.ingest_events` (
  event_id    STRING NOT NULL,
  chunk_id    STRING NOT NULL,
  tenant_id   STRING NOT NULL,
  event_type  STRING NOT NULL OPTIONS(description = 'created | updated | reindexed | withheld'),
  reason      STRING,
  occurred_at TIMESTAMP NOT NULL
)
PARTITION BY DATE(occurred_at)
CLUSTER BY tenant_id, event_type
""")
print("chunk_metadata and ingest_events ready")

## Cell 1: The source, and a corpus worth measuring
Two decisions here, and the second is the interesting one.

Lesson 5.3's `doc_chunks` carries four columns - chunk_id, doc_id, chunk_text, doc_type - because that is all its retrieval demo needed. Feature engineering needs a heading path, a revision date, a tenant and page numbers, which in production come from lesson 4.1. The view normalises what exists and **synthesises** the rest, with `provenance_synthetic` so nothing downstream confuses them.

Those twelve chunks are also **clean** - every one a well-formed English policy clause with no personal data, no tables and no figures. That makes them a good retrieval demo and a useless feature-engineering demo: measure a clean corpus and every feature is constant, the PII gate finds nothing, and the router has a single class to predict.

> So this cell adds **ten deliberately hard chunks**: an 8-token page header, a policy superseded in 2023, a clause carrying a tax id, a Hindi passage, a markdown table, a figure caption, and clauses that reference other clauses. The identifiers are documentation placeholders, never real data. `is_hard_case` marks them, and dropping the union is the only change needed when 12.5 lands the real pipeline.

In [ ]:
# Lesson 5.3 seeded `doc_chunks` with twelve clean ACME policy clauses. Confirm it is there before
# doing anything else - a missing upstream fixture should say so, not fail six cells later.
_have = list(bq.query(f"""
  SELECT COUNT(*) AS n FROM `{PROJECT_ID}.{DATASET}.__TABLES_SUMMARY__`
  WHERE table_id = 'doc_chunks'
""").result())[0].n
assert _have, (f"`{PROJECT_ID}.{DATASET}.doc_chunks` is missing. Run lesson 5.3's fixture cell first "
               "- it seeds the twelve ACME clauses this lesson features.")

# Those twelve chunks are CLEAN: every one is a well-formed policy clause, in English, with no
# personal data, no tables and no figures. That is what makes them a good RAG demo and a useless
# feature-engineering demo. Measure a clean corpus and every feature comes back constant, the PII
# gate finds nothing, and the router has one class - and you learn nothing about any of them.
#
# So the lesson adds ten chunks that are deliberately hard. Every one is a shape that has cost a
# real team a real outage: a page header that is 8 tokens of nothing, a policy superseded three
# years ago that still ranks, a clause carrying a tax id, a Hindi passage in an English index.
# The identifiers below are documentation placeholders, not data - ABCDE1234F is the format
# example from the tax department's own guidance, and acme.in is not a real domain.
HARD_CASES = [
    # (chunk_id, doc_id, chunk_text, doc_type)
    ("hx-01", "acme_hr_handbook", "ACME Confidential. Page 14 of 87.", "hr_policy"),
    ("hx-02", "acme_hr_handbook",
     "LV-05 Leave Policy (2023 edition). Earned leave accrued at 1.2 days per month of service. "
     "This clause is superseded by LV-01 with effect from 1 April 2024 and is retained for "
     "reference only.", "hr_policy"),
    ("hx-03", "acme_finance_manual",
     "EXP-40 Contractor Records. Tax records held on file for retainer contractors. Example row: "
     "vendor Sharma Consulting, PAN ABCDE1234F, engaged under VC-09.", "finance_policy"),
    ("hx-04", "acme_hr_handbook",
     "HR-88 Emergency Escalation. Outside business hours contact the HR duty officer at "
     "priya.sharma@acme.in or on +91 9876543210. Do not route emergencies through the ticketing "
     "system.", "hr_policy"),
    ("hx-05", "acme_hr_handbook",
     "LV-01 अवकाश नीति। अर्जित अवकाश प्रति माह 1.5 दिन की दर से जमा होता है। "
     "अप्रयुक्त अवकाश अगले वर्ष में आगे ले जाया जा सकता है।", "hr_policy"),
    ("hx-06", "acme_finance_manual",
     "EXP-13 Per Diem Grades. | Grade | Metro | Non-metro |\n| E1-E3 | Rs 2,500 | Rs 1,800 |\n"
     "| M1+ | Rs 3,500 | Rs 2,600 |\nRates are revised each financial year.", "finance_policy"),
    ("hx-07", "acme_it_manual",
     "IT-SEC-09 Access Review. Figure 3 shows the quarterly access review workflow from request "
     "through approval to revocation. Reviewers must complete each cycle before the quarter "
     "closes.", "it_policy"),
    ("hx-08", "acme_legal_manual",
     "VC-11 Vendor Onboarding. Vendor onboarding follows the process in VC-09 and requires the "
     "security review described under IT-SEC-04 before any data is shared.", "legal_policy"),
    ("hx-09", "acme_hr_handbook",
     "CUL-01 Working Principles. We assume good intent, we write things down, and we prefer a "
     "short conversation to a long thread. Managers are expected to model this rather than "
     "announce it.", "hr_policy"),
    ("hx-10", "acme_hr_handbook",
     "HR-91 Statutory Records. Provident fund enrolment requires the employee identity number on "
     "file, for example 1234 5678 9012, together with the bank mandate.", "hr_policy"),
]

_tbl = f"{PROJECT_ID}.{DATASET}.chunk_hard_cases"
_schema = [bigquery.SchemaField(n, "STRING") for n in ("chunk_id", "doc_id", "chunk_text", "doc_type")]
bq.delete_table(_tbl, not_found_ok=True)
bq.create_table(bigquery.Table(_tbl, schema=_schema))
_errs = bq.insert_rows(bq.get_table(_tbl), HARD_CASES)
assert not _errs, _errs
print(f"  seeded {len(HARD_CASES)} deliberately hard chunks")

# The view is the seam. It normalises the four columns 5.3 supplies into the shape feature
# engineering needs, synthesises what the fixture cannot supply, and says which is which.
#
# In production the missing columns come from lesson 4.1's Document AI output, which really does
# carry ancestor headings, source_uri and page numbers. Never let synthesised provenance reach a
# production table without a column announcing it.
PREPARE = f"""
CREATE OR REPLACE VIEW `{PROJECT_ID}.{DATASET}.chunk_source` AS
WITH raw AS (
  SELECT CAST(chunk_id AS STRING) AS chunk_id, doc_id, chunk_text, doc_type, FALSE AS is_hard_case
  FROM `{PROJECT_ID}.{DATASET}.doc_chunks`
  UNION ALL
  SELECT chunk_id, doc_id, chunk_text, doc_type, TRUE AS is_hard_case
  FROM `{PROJECT_ID}.{DATASET}.chunk_hard_cases`
)
SELECT
  chunk_id,
  '{TENANT}'                     AS tenant_id,       -- both fixtures are single-tenant
  chunk_text                     AS text,
  doc_type                       AS doc_type_seed,   -- the fixture's own label; Step 4 predicts its own
  CONCAT('gs://documind-acme/', doc_id, '.pdf') AS source_uri,
  is_hard_case,

  -- REAL(ish): 4.1 keeps the ancestor headings. Both fixtures encode them in the clause code that
  -- opens each passage ('LV-07 Leave Encashment. ...'), so recover that much and no more. The
  -- code may itself contain letters between hyphens, which is why IT-SEC-04 needs the middle group.
  REGEXP_EXTRACT(chunk_text, r'^([A-Z]{{2,}}(?:-[A-Z]{{2,}})*-[0-9]{{2,}}[^.]*)') AS heading_path,

  -- SYNTHETIC: neither fixture has a revision date. Derive a stable pseudo-date from the chunk id
  -- so freshness_days varies and the filters in Step 7 have something to bite on. The superseded
  -- 2023 clause is pinned old on purpose, because that is the case the filter has to catch.
  CASE WHEN chunk_id = 'hx-02' THEN DATE '2023-06-30'
       ELSE DATE_SUB(CURRENT_DATE(),
                     INTERVAL MOD(ABS(FARM_FINGERPRINT(chunk_id)), 900) DAY) END AS last_revised_at,
  TRUE                           AS provenance_synthetic,

  CAST(NULL AS INT64)            AS page_start,
  CAST(NULL AS INT64)            AS page_end
FROM raw
"""
run(PREPARE)

for r in run(f"""
  SELECT COUNT(*) AS chunks,
         COUNTIF(is_hard_case) AS hard,
         COUNTIF(heading_path IS NOT NULL) AS with_heading,
         MIN(last_revised_at) AS oldest, MAX(last_revised_at) AS newest
  FROM `{PROJECT_ID}.{DATASET}.chunk_source`
"""):
    print(f"  {r.chunks} chunks ({r.hard} hard cases) | {r.with_heading} with a recovered heading")
    print(f"  revision dates span {r.oldest} to {r.newest} (synthetic - see provenance_synthetic)")

# Everything downstream reads chunk_source, not doc_chunks. When lesson 12.5 lands the real
# ingestion pipeline, only this view changes: the feature SQL, the filters, the router and the
# gate are already written against the shape they will get.

## Cell 2: The features that cost nothing
Arithmetic and regular expressions over text you already have — no model call, so this runs over the whole corpus for the price of a scan.

> **Read where the patterns are defined.** They are named constants, then interpolated. Inside an f-string, `{5}` is a replacement field: Python evaluates it and emits the digit 5, so `[A-Z]{5}` silently becomes `[A-Z]5` — still valid, still runs, matches nothing. This is a real bug that was caught while writing this lesson, and Cell 4 asserts the patterns before trusting any count.

In [ ]:
# Every feature here is arithmetic or a regular expression over text you already have.
# No model call, so this runs over the whole corpus for the price of a query scan.
#
# The patterns live in their own constants, NOT inline in the f-string below. Inside an f-string a
# regex quantifier like {5} is a replacement field: Python evaluates it and silently emits the
# digit 5 instead of the quantifier. The regex still runs, matches nothing, and reports a clean
# result. An interpolated value is not re-scanned, so the braces survive.
TABLE_RE      = r'(?i)\|.*\|.*\||\btable\s+\d'
FIGURE_RE     = r'(?i)\bfigure\s+\d|\bfig\.\s*\d'
DEVANAGARI_RE = r'[\x{0900}-\x{097F}]'     # the Unicode block: a range test, not a model call
LATIN_WORD_RE = r'[A-Za-z]{4,}'

# Prove them before trusting them. This is the same discipline the PII gate uses in Step 5, applied
# here because a silently-broken feature is only quieter than a silently-broken control, not safer:
# has_table that is always FALSE looks exactly like a corpus with no tables.
import re as _re
assert _re.search(TABLE_RE, '| Grade | Metro |'),      'table pattern is broken'
assert _re.search(FIGURE_RE, 'Figure 3 shows the'),    'figure pattern is broken'
assert _re.search(LATIN_WORD_RE, 'accrues monthly'),   'latin-word pattern is broken'
assert not _re.search(FIGURE_RE, 'configure the vpn'), 'figure pattern over-matches'

# DEVANAGARI_RE is deliberately NOT asserted here, and the reason is worth more than the assert.
# `\x{0900}` is RE2's braced hex escape. BigQuery speaks RE2; Python's `re` does not, and rejects
# the pattern outright. So a Python self-test cannot cover it, and pretending otherwise by
# rewriting the pattern for the test would prove the test's copy, not the shipped one.
# Verify it by its effect instead: the query below must report one 'hi' chunk. When a check cannot
# reach the code, move the check to the result.
try:
    _re.compile(DEVANAGARI_RE); _reachable = True
except _re.error:
    _reachable = False
assert not _reachable, ("DEVANAGARI_RE compiled in Python, so it is no longer the RE2 form "
                        "BigQuery needs - check the escape")
print('  pattern self-test passed (3 asserted in Python, 1 verified by its result below)')

CHEAP = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.chunk_features_cheap` AS
SELECT
  chunk_id,
  tenant_id,
  source_uri,
  page_start,
  page_end,

  -- Approximate, and deliberately so: an exact count means an API call per chunk.
  -- Lesson 2.1 measured about 4 characters per token for English. Hindi runs richer, so this
  -- UNDER-counts Devanagari text, and under-counting a budget is the safe direction.
  CAST(CEIL(LENGTH(text) / 4) AS INT64)                    AS token_count,

  -- How deep in the document's heading tree this chunk sits. 4.1's Layout Parser keeps the
  -- ancestor headings, so counting the separators gives the depth.
  ARRAY_LENGTH(SPLIT(COALESCE(heading_path, ''), ' > ')) - 1  AS heading_depth,

  REGEXP_CONTAINS(text, r'{TABLE_RE}')                     AS has_table,
  REGEXP_CONTAINS(text, r'{FIGURE_RE}')                    AS has_figure,

  CASE
    WHEN REGEXP_CONTAINS(text, r'{DEVANAGARI_RE}')
     AND REGEXP_CONTAINS(text, r'{LATIN_WORD_RE}')         THEN 'mixed'
    WHEN REGEXP_CONTAINS(text, r'{DEVANAGARI_RE}')         THEN 'hi'
    ELSE 'en'
  END                                                          AS language,

  DATE_DIFF(CURRENT_DATE(), DATE(last_revised_at), DAY)        AS freshness_days

FROM `{PROJECT_ID}.{DATASET}.chunk_source`
WHERE tenant_id = '{TENANT}'
"""
run(CHEAP)

for r in run(f"""
  SELECT language, COUNT(*) AS chunks, ROUND(AVG(token_count)) AS avg_tokens,
         COUNTIF(has_table) AS with_tables, ROUND(AVG(freshness_days)) AS avg_age_days
  FROM `{PROJECT_ID}.{DATASET}.chunk_features_cheap`
  GROUP BY language ORDER BY chunks DESC
"""):
    print(f"  {r.language:6} {r.chunks:>4} chunks | avg {r.avg_tokens:>4} tokens | "
          f"{r.with_tables:>3} with tables | avg age {r.avg_age_days} days")

# The Python assert above could not reach DEVANAGARI_RE, so the check moves to the result. If the
# pattern had been corrupted by interpolation this query would report only 'en', and the corpus
# would look monolingual while containing Hindi - a silent feature failure of exactly the kind
# Step 1 promised to make visible.
_langs = {r.language: r.chunks for r in run(f"""
  SELECT language, COUNT(*) AS chunks FROM `{PROJECT_ID}.{DATASET}.chunk_features_cheap`
  GROUP BY language
""")}
assert _langs.get('hi'), ("no Hindi chunk detected, so DEVANAGARI_RE did not survive "
                          "interpolation - this is the check the Python assert could not make")
print(f"  DEVANAGARI_RE verified by result: {_langs['hi']} Hindi chunk(s) found")

## Cell 3: The feature that costs a model call
`AI.GENERATE_TABLE` classifies the whole corpus in one statement.

It is a **table-valued** function, unlike lesson 5.3's scalar `AI.GENERATE`: it takes a model, a table or subquery of inputs, and an `output_schema` STRING, and returns a table with those columns — so it goes in the `FROM` clause. Each field may carry an `OPTIONS(description = …)` that the model reads as part of the instruction.

> **Why Flash-Lite, why nightly.** This is one model call per chunk over the entire corpus — the highest-volume call in Module 5 and one of the easiest tasks. A document's type does not change between two questions, so paying for it on the query path is waste. Lesson 5.4's scheduled query is where this belongs.

Note the prompt forbids judging from the file name. Left alone the model reads `MSA_final_v3.pdf` and answers “contract” without reading the passage — a filename heuristic wearing a model's confidence. `doc_type_conf` is how you find those.

In [ ]:
# ONE remote model, reused. AI.GENERATE_TABLE is a table-valued function: it takes a model,
# a table (or subquery) of inputs, and an output_schema STRING, and returns a table with exactly
# those columns. Verified against the AI.GENERATE_TABLE reference on 2026-09-03.
run(f"""
CREATE MODEL IF NOT EXISTS `{PROJECT_ID}.{DATASET}.classifier_model`
REMOTE WITH CONNECTION DEFAULT
OPTIONS (ENDPOINT = '{CLASSIFY_MODEL}')
""")

# Classify every chunk in ONE statement. This is the expensive feature - one model call per row -
# so it runs nightly over new chunks only, not on every query.
CLASSIFY = rf"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.chunk_features_doctype` AS
SELECT chunk_id, doc_type, doc_type_conf
FROM AI.GENERATE_TABLE(
  MODEL `{PROJECT_ID}.{DATASET}.classifier_model`,
  (
    SELECT
      chunk_id,
      CONCAT(
        'Classify this passage from an enterprise document. ',
        'Answer with one doc_type from: policy, contract, report, form, correspondence. ',
        'doc_type_conf is your confidence from 0 to 1. ',
        'Judge only from the passage; do not guess from the file name.\n\nPassage:\n',
        SUBSTR(text, 1, 2000)
      ) AS prompt
    FROM `{PROJECT_ID}.{DATASET}.chunk_source`
    WHERE tenant_id = '{TENANT}'
  ),
  STRUCT(
    "doc_type STRING OPTIONS(description = 'One of: policy, contract, report, form, correspondence'), "
    "doc_type_conf FLOAT64 OPTIONS(description = 'Confidence from 0 to 1')" AS output_schema
  )
)
"""
run(CLASSIFY)

for r in run(f"""
  SELECT doc_type, COUNT(*) AS n, ROUND(AVG(doc_type_conf), 3) AS avg_conf,
         COUNTIF(doc_type_conf < 0.7) AS low_confidence
  FROM `{PROJECT_ID}.{DATASET}.chunk_features_doctype`
  GROUP BY doc_type ORDER BY n DESC
"""):
    print(f"  {r.doc_type:15} {r.n:>4} chunks | avg confidence {r.avg_conf} | "
          f"{r.low_confidence} below 0.7")

## Cell 4: The gate that is not a ranking signal
PII does not lower a chunk's score. It keeps the chunk **out of the index**.

> The tempting design is `WHERE NOT pii_flag` on the retrieval query. It works until one code path forgets it — and lesson 4.6 showed what a forgotten filter does in a multi-tenant system. The design that holds is subtraction: a flagged chunk is never written as a datapoint at all, so there is no query to get wrong.

The pattern self-test runs **before** the count. A privacy control that fails silently is worse than none, because it manufactures confidence.

In [ ]:
# The PII flag is not a feature you rank on. It is a gate: TRUE keeps a chunk OUT of the shared
# index entirely. Two passes, cheapest first.
#
# Same rule as Step 3: the patterns are constants, never written inline in the f-string. A
# quantifier like {5} inside an f-string becomes the digit 5, and the detector then finds nothing
# while reporting success - the most dangerous shape a bug can take in a privacy control.
PAN_RE     = r'\b[A-Z]{5}[0-9]{4}[A-Z]\b'          # ABCDE1234F
AADHAAR_RE = r'\b\d{4}\s?\d{4}\s?\d{4}\b'          # 12 digits, usually spaced in threes
MOBILE_RE  = r'\b(?:\+91[\-\s]?)?[6-9]\d{9}\b'     # Indian mobile, optional +91
EMAIL_RE   = r'[\w.\-]+@[\w\-]+\.[A-Za-z]{2,}'

# Prove the patterns survived interpolation BEFORE trusting any count they produce.
import re as _re
assert _re.search(PAN_RE, 'PAN ABCDE1234F on file'),        'PAN pattern is broken'
assert _re.search(AADHAAR_RE, 'UID 1234 5678 9012'),        'Aadhaar pattern is broken'
assert _re.search(MOBILE_RE, 'call +91 9876543210'),        'mobile pattern is broken'
assert _re.search(EMAIL_RE, 'priya.sharma@acme.in'),        'email pattern is broken'
assert not _re.search(PAN_RE, 'policy NP-03 clause 4'),     'PAN pattern over-matches'
print('  pattern self-test passed')

PII = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.chunk_features_pii` AS
SELECT
  chunk_id,
  -- Pass 1: deterministic patterns for the identifiers this corpus actually contains.
  (   REGEXP_CONTAINS(text, r'{PAN_RE}')
   OR REGEXP_CONTAINS(text, r'{AADHAAR_RE}')
   OR REGEXP_CONTAINS(text, r'{MOBILE_RE}')
   OR REGEXP_CONTAINS(text, r'{EMAIL_RE}')
  ) AS pii_flag
FROM `{PROJECT_ID}.{DATASET}.chunk_source`
WHERE tenant_id = '{TENANT}'
"""
run(PII)

for r in run(f"""
  SELECT COUNTIF(pii_flag) AS withheld, COUNT(*) AS total,
         ROUND(100 * COUNTIF(pii_flag) / COUNT(*), 1) AS pct
  FROM `{PROJECT_ID}.{DATASET}.chunk_features_pii`
"""):
    print(f"  {r.withheld} of {r.total} chunks withheld ({r.pct}%)")

# Pass 2 belongs in production and is NOT run here: Sensitive Data Protection - the current name
# for Cloud DLP, though the API is still the DLP API - inspects the table with more than 100
# built-in detectors and reports findings cell by cell. A regular expression finds the identifiers
# you predicted; the inspection service finds the ones you did not. Ship both, in that order: the
# cheap pass keeps the expensive one's bill down.

## Cell 5: Assemble, and write down what happened
The merge is idempotent — re-running the nightly job updates rows rather than duplicating them, the same property lesson 4.6 needed from its stable node ids. The insert into `ingest_events` is **not** idempotent, and must not be: it is a log, and two runs really did happen.

In [ ]:
# Join the three feature sets into chunk_metadata, and log what happened.
MERGE = f"""
MERGE `{PROJECT_ID}.{DATASET}.chunk_metadata` T
USING (
  SELECT c.chunk_id, c.tenant_id, c.source_uri, c.page_start, c.page_end,
         c.token_count, c.heading_depth, c.has_table, c.has_figure, c.language, c.freshness_days,
         d.doc_type, d.doc_type_conf, p.pii_flag,
         CURRENT_TIMESTAMP() AS featured_at
  FROM `{PROJECT_ID}.{DATASET}.chunk_features_cheap` c
  LEFT JOIN `{PROJECT_ID}.{DATASET}.chunk_features_doctype` d USING (chunk_id)
  LEFT JOIN `{PROJECT_ID}.{DATASET}.chunk_features_pii`     p USING (chunk_id)
) S
ON T.chunk_id = S.chunk_id AND T.tenant_id = S.tenant_id
WHEN MATCHED THEN UPDATE SET
  token_count = S.token_count, heading_depth = S.heading_depth, has_table = S.has_table,
  has_figure = S.has_figure, language = S.language, freshness_days = S.freshness_days,
  doc_type = S.doc_type, doc_type_conf = S.doc_type_conf, pii_flag = S.pii_flag,
  featured_at = S.featured_at
WHEN NOT MATCHED THEN INSERT ROW
"""
run(MERGE)

# The append-only log. 'withheld' is the row an auditor will ask about.
run(f"""
INSERT INTO `{PROJECT_ID}.{DATASET}.ingest_events`
  (event_id, chunk_id, tenant_id, event_type, reason, occurred_at)
SELECT GENERATE_UUID(), chunk_id, tenant_id,
       IF(pii_flag, 'withheld', 'reindexed'),
       IF(pii_flag, 'pii_flag=true', 'features recomputed'),
       CURRENT_TIMESTAMP()
FROM `{PROJECT_ID}.{DATASET}.chunk_metadata`
WHERE tenant_id = '{TENANT}'
""")
# index_feed is what the upsert job reads: the chunks that are ALLOWED into the shared index.
# Building it by subtraction is the control from Step 5; the quality rule in Step 10 is the
# regression test that the subtraction still happens.
run(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.index_feed` AS
SELECT chunk_id, tenant_id, doc_type, language, token_count, freshness_days, heading_depth, has_table
FROM `{PROJECT_ID}.{DATASET}.chunk_metadata`
WHERE tenant_id = '{TENANT}'
  AND NOT COALESCE(pii_flag, TRUE)      -- unknown counts as unsafe
  AND token_count >= 32                 -- too short to carry an answer (Step 1's page header)
""")

for r in run(f"""
  SELECT
    (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET}.chunk_metadata`) AS in_corpus,
    (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET}.index_feed`)     AS in_feed
"""):
    print(f"  corpus {r.in_corpus} chunks -> index feed {r.in_feed} "
          f"({r.in_corpus - r.in_feed} withheld by subtraction)")

# Read that gap. It is not waste, it is the whole control surface: every chunk in it was removed
# by a WHERE clause in one place, rather than by a filter each retrieval path has to remember.

## Cell 6: Features become filters
This is the payoff. `restricts` are tokens matched exactly; `numeric_restricts` are numbers the query compares with an operator (`LESS`, `LESS_EQUAL`, `EQUAL`, `GREATER_EQUAL`, `GREATER`).

> **The logic is AND across namespaces, OR within one.** So `doc_type: [policy, contract]` with `tenant: [acme]` reads “acme AND (policy OR contract)”. That single rule is your namespace design: alternatives you want to widen go in one namespace, constraints you want to stack go in separate ones. Putting tenant in the same namespace as anything else breaks isolation — and nothing errors.

In [ ]:
import json

def to_datapoint(row, embedding, sparse=None) -> dict:
    """One chunk_metadata row plus its vector -> one Vector Search datapoint with filters attached.

    restricts are TOKENS: filter with allow/deny lists, matched exactly.
    numeric_restricts are NUMBERS: the query supplies the comparison operator.

    This is the INDEX INPUT format - the JSON you write to Cloud Storage for a batch index build:
    id / embedding / restricts[{namespace, allow, deny}] / numeric_restricts / sparse_embedding.
    The streaming upsertDatapoints REST call spells the same fields differently (datapointId,
    featureVector, allowList, denyList). Pick one surface and stay on it; a datapoint that mixes
    the two is rejected by both, and the error does not say which half is wrong.
    Verified against the Vector Search filtering reference on 2026-09-03.

    The embedding arrives as an argument because chunk_metadata deliberately holds no vectors
    (Step 2): the upsert job joins each feature row to the vector it has just computed.
    """
    dp = {
        "id": row["chunk_id"],
        "embedding": [float(x) for x in embedding],         # 768-d, gemini-embedding-001
        "restricts": [
            {"namespace": "tenant",    "allow": [row["tenant_id"]]},
            {"namespace": "doc_type",  "allow": [row["doc_type"] or "unknown"]},
            {"namespace": "language",  "allow": [row["language"]]},
            {"namespace": "has_table", "allow": ["yes" if row["has_table"] else "no"]},
        ],
        "numeric_restricts": [
            {"namespace": "token_count",    "value_int": int(row["token_count"])},
            {"namespace": "freshness_days", "value_int": int(row["freshness_days"])},
            {"namespace": "heading_depth",  "value_int": int(row["heading_depth"])},
        ],
    }
    if sparse:
        dp["sparse_embedding"] = sparse                     # Step 8 builds this
    return dp

# A query filter. Note the two logics: AND across namespaces, OR within one namespace.
# So this reads: tenant is acme, AND doc_type is policy or contract, AND revised within 400 days.
QUERY_FILTER = {
    "restricts": [
        {"namespace": "tenant",   "allow": ["acme"]},
        {"namespace": "doc_type", "allow": ["policy", "contract"]},
    ],
    "numeric_restricts": [
        {"namespace": "freshness_days", "value_int": 400, "op": "LESS_EQUAL"},
    ],
}
print(json.dumps(QUERY_FILTER, indent=2))

# PII-flagged chunks are never written as datapoints at all. A filter you can forget to apply is
# not a control; leaving the row out of the index is.

### Why filtering at the index beats filtering after it

In [ ]:
# Why filter at the index instead of after retrieval.
def compare(corpus: int, matching_pct: float, top_k: int = 20):
    matching = int(corpus * matching_pct)
    print(f"corpus {corpus:,} chunks, {matching_pct:.0%} match the filter")
    print(f"  filter AFTER retrieval : ask for {top_k}, expect ~{top_k * matching_pct:.1f} usable "
          f"-> to get {top_k} you must fetch ~{int(top_k / matching_pct):,}")
    print(f"  filter AT the index    : the search runs over {matching:,} candidates, "
          f"returns {top_k} usable")

for pct in (0.5, 0.2, 0.05):
    compare(100_000, pct)
    print()

# The narrower the filter, the worse post-filtering gets - and a tenant filter is the narrowest
# of all. Post-filtering a multi-tenant index is also the failure mode 4.6 called a leak: a
# forgotten WHERE clause returns another tenant's chunk. A restrict cannot be forgotten at
# query time, because a datapoint without the tenant namespace never matches.

## Cell 7: A sparse vector beside the dense one
Lesson 4.5 ran BM25 in the notebook process and fused two rankings there — the right way to learn it, the wrong way to run it, because the sparse index dies with the kernel. Here the sparse weights go into the **same** Vector Search index as the dense vectors, so the fusion happens where the corpus lives. The kit's own indexer writes no sparse vector today (12 September 2026): `hybrid.py` hashes the query's tokens, but every datapoint carries the dense vector only, so `RETRIEVAL_MODE=hybrid` runs its dense half on the full profile, and the lean Firestore profile refuses the mode at startup rather than report a fusion it cannot do. This cell is what 12.5's indexer would write for the fusion to be real.

A record needs at least one of `embedding` or `sparse_embedding`, and may carry both. The `dimensions` array is what makes it sparse: only the non-zero positions are stored, and each integer is a term id.

In [ ]:
from rank_bm25 import BM25Okapi
import re, json

def tok(s: str) -> list:
    return re.findall(r"[a-z0-9\-]+", s.lower())

def sparse_vectors(corpus_texts: list) -> list:
    """BM25 weights as Vector Search sparse embeddings, one per document.

    Format verified 2026-09-03: {"values": [...], "dimensions": [...]}, where each dimension is the
    integer term id holding a non-zero value. A record needs at least one of embedding or
    sparse_embedding and may carry both - which is what lets ONE index serve hybrid search.

    Scoring: BM25 is a query-document score, so a term's weight in a document is that document's
    score for the single-term query [term]. Compute it ONCE per term across the whole corpus -
    get_scores returns every document at once - rather than once per document-term pair. The naive
    nesting costs O(vocabulary x documents) calls instead of O(vocabulary), and on a real corpus it
    does not finish.

    Index by POSITION, never by value: two chunks with identical text are still two rows.
    """
    docs = [tok(t) for t in corpus_texts]
    bm25 = BM25Okapi(docs)
    vocab = sorted({term for d in docs for term in d})
    contains = [set(d) for d in docs]

    per_doc = [dict() for _ in docs]
    for term_id, term in enumerate(vocab):
        scores = bm25.get_scores([term])
        for doc_i, s in enumerate(scores):
            # keep only positive weights, and only for terms the document actually contains
            if s > 0 and term in contains[doc_i]:
                per_doc[doc_i][term_id] = float(s)

    out = []
    for scores in per_doc:
        dims = sorted(scores)
        out.append({"values": [scores[k] for k in dims], "dimensions": dims})
    return out

# A datapoint carrying both vectors. Lesson 4.5 fused dense and sparse inside the notebook process,
# which is right for twelve chunks and wrong for a corpus: that sparse index dies with the kernel.
# Writing the sparse half into the index puts the fusion where the corpus lives.
# Take the chunks the gate ALLOWED - index_feed, not chunk_metadata - and build real datapoints
# for the first three. index_feed carries the features; chunk_source carries the text.
feed = run(f"""
  SELECT f.chunk_id, f.tenant_id, f.doc_type, f.language, f.token_count, f.freshness_days,
         f.heading_depth, f.has_table, s.content
  FROM `{PROJECT_ID}.{DATASET}.index_feed` f
  JOIN `{PROJECT_ID}.{DATASET}.chunk_source` s USING (chunk_id)
  ORDER BY f.chunk_id LIMIT 3
""")

# One embedding call for the batch, pinned to 768 dimensions. RETRIEVAL_DOCUMENT is the corpus-side
# task type from lesson 2.4: a query embedded later must use RETRIEVAL_QUERY, or the two sit in
# different halves of the space and every distance you measure is meaningless.
resp = emb_client.models.embed_content(
    model=EMBED_MODEL,
    contents=[r.content for r in feed],
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT",
                                    output_dimensionality=EMBED_DIMS),
)
dense  = [e.values for e in resp.embeddings]
sparse = sparse_vectors([r.content for r in feed])

datapoints = [to_datapoint(dict(r), d, s) for r, d, s in zip(feed, dense, sparse)]
assert all(len(dp["embedding"]) == EMBED_DIMS for dp in datapoints), \
    f"expected {EMBED_DIMS}-d vectors; the output_dimensionality pin did not hold"

_show = dict(datapoints[0])
_show["embedding"] = f"[{len(_show['embedding'])} floats]"
print(json.dumps(_show, indent=2, ensure_ascii=False)[:1200])
print(f"\n  {len(datapoints)} datapoints built | sparse terms each:",
      [len(dp.get("sparse_embedding", {}).get("values", [])) for dp in datapoints])

# ---------------------------------------------------------------------------------------------
# Run it on three documents and read the result carefully - it is not what most people expect.
_demo = ["notice period is 60 days", "notice period is 60 days", "earned leave accrues monthly"]
_vecs = sparse_vectors(_demo)
print("\nnon-zero terms per document:", [len(v["values"]) for v in _vecs])

_bm = BM25Okapi([tok(t) for t in _demo])
for _term in ("notice", "earned"):
    _n = sum(1 for t in _demo if _term in tok(t))
    print(f"  '{_term}' appears in {_n}/3 documents | idf = {_bm.idf.get(_term, 0):+.4f}")

print("""
  The first two documents come back EMPTY, and that is BM25 working correctly.
  Okapi's inverse document frequency goes NEGATIVE for a term that appears in more than half the
  corpus: a word in almost every document cannot help you tell documents apart, so it earns no
  weight. On a three-document corpus, 'notice' is in two of them and scores below zero, while
  'earned' is in one and scores positively.

  Two consequences worth carrying into production:
    1. Do not evaluate sparse retrieval on a toy corpus. The statistic that makes BM25 useful is
       a property of a LARGE vocabulary, and it inverts on a small one.
    2. An empty sparse vector is not an error. The datapoint still indexes on its dense vector -
       a record needs at least one of the two - so the chunk stays retrievable and simply
       contributes nothing to the keyword half of the fusion.""")

### First, labels
A supervised model needs labels, and labels are the part nobody budgets for. On a real corpus you get these from lesson 4.7's golden set: run each question down each path, and the path that retrieved the required chunk is that chunk's label.

> The heuristic below is a **stand-in** so the training cell runs today. A classifier fitted to heuristic labels has learned the heuristic, not the routing.

In [ ]:
# The router trained below is SUPERVISED: it needs labels, and labels are the part nobody budgets for.
#
# A label here answers "which retrieval path serves this chunk best?" - vector for self-contained
# prose, graph for a clause whose value is what it points at, hybrid when an exact token has to
# match and a dense vector will smudge it. On a real corpus you get these from lesson 4.7's golden
# set: run each question down each path, and the path that retrieved the required chunk is the
# label. The heuristic below is a STAND-IN so the training cell runs today.
#
# One subtlety worth the line it costs. Every chunk in this corpus OPENS with its own clause code,
# so a naive "contains a clause code" test labels all 22 the same way and the classifier gets a
# single class to predict - which BigQuery ML rejects outright. Strip the heading first and ask
# what the BODY contains. A code in the heading is this chunk's name; a code in the body is a
# reference to a different chunk, and that is the thing that makes the graph path worth its cost.
HEADING_RE   = r'^[A-Z]{2,}(?:-[A-Z]{2,})*-[0-9]{2,}[^.]*\.\s*'
CROSSREF_RE  = r'\b[A-Z]{2,}(?:-[A-Z]{2,})*-[0-9]{2,}\b'
EXACTMATCH_RE = r'\bRs\.?\s?[0-9,]{3,}|[0-9]+(?:\.[0-9]+)?\s?%|\b[0-9]{1,3}\s?(?:days|months|years)\b'

import re as _re
assert _re.sub(HEADING_RE, '', 'IT-SEC-04 Device Security. All laptops.') == 'All laptops.', \
    'heading pattern misses a multi-part clause code'
assert _re.search(CROSSREF_RE, 'follows VC-09 and'),        'cross-reference pattern is broken'
assert _re.search(EXACTMATCH_RE, 'per diem is Rs 2,500'),   'exact-match pattern is broken'
assert not _re.search(EXACTMATCH_RE, 'we write things down'), 'exact-match pattern over-matches'
print('  routing pattern self-test passed')

LABELS = rf"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.route_labels` AS
WITH body AS (
  SELECT chunk_id, REGEXP_REPLACE(text, r'{HEADING_RE}', '') AS body
  FROM `{PROJECT_ID}.{DATASET}.chunk_source`
)
SELECT
  chunk_id,
  CASE
    -- a clause that names ANOTHER clause is relational: the graph path (4.6) earns its cost
    WHEN REGEXP_CONTAINS(body, r'{CROSSREF_RE}')    THEN 'graph'
    -- an exact amount, rate or duration is what sparse retrieval catches and dense retrieval blurs
    WHEN REGEXP_CONTAINS(body, r'{EXACTMATCH_RE}')  THEN 'hybrid'
    ELSE 'vector'
  END AS route
FROM body
"""
run(LABELS)

_dist = {r.route: r.n for r in run(f"""
  SELECT route, COUNT(*) AS n FROM `{PROJECT_ID}.{DATASET}.route_labels`
  GROUP BY route ORDER BY n DESC
""")}
for k, v in _dist.items():
    print(f"  {k:8} {v:>3} chunks")
assert len(_dist) >= 2, ("the label heuristic produced a single class, and logistic_reg needs at "
                         "least two. Check that HEADING_RE still strips this corpus's codes.")

print("""
  Two things to carry out of this cell.
  First, the assert. A single-class label set is not a bad model, it is a model that cannot be
  trained, and finding that out from a clear message beats finding it out from BigQuery.
  Second, these labels are heuristic. A classifier fitted to a regular expression has learned the
  regular expression - and it will report respectable accuracy while doing it, which is exactly why
  lesson 5.1 warned that a metric from a small fixture is a training-set metric. Replace these with
  measured labels from 4.7's golden set before you believe any number the next cell prints.""")

## Cell 8: A router that costs a lookup, not a model call
`TRANSFORM` is the point. The preprocessing declared inside `CREATE MODEL` is applied automatically by `ML.PREDICT` and `ML.EVALUATE`, so you never write the transform again at serving time — which means you cannot write it *differently*. That difference is training-serving skew, and it is one of the least visible ways a model degrades after deployment.

> `auto_class_weights = TRUE` because the routes are not balanced. Without it the majority route wins on volume and the classifier looks accurate while being useless.

In [ ]:
# Lesson 3.3 routed on complexity using a Flash-Lite call per query. That is one model call
# before every real model call. Here the routing decision is learned from features you already
# compute, so it costs a table lookup.
#
# TRANSFORM is the point: the preprocessing declared here is stored WITH the model and applied
# automatically by ML.PREDICT and ML.EVALUATE. There is no separate serving-time transform to
# keep in sync, which is the usual source of training-serving skew.
ROUTER = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.retrieval_router`
TRANSFORM (
  ML.QUANTILE_BUCKETIZE(token_count, 4)    OVER () AS token_bucket,
  ML.QUANTILE_BUCKETIZE(freshness_days, 4) OVER () AS freshness_bucket,
  CAST(has_table AS STRING)                        AS has_table_s,
  CAST(heading_depth AS STRING)                    AS heading_depth_s,
  doc_type,
  language,
  route
)
OPTIONS (
  model_type = 'logistic_reg',
  input_label_cols = ['route'],
  auto_class_weights = TRUE          -- the classes are not balanced; do not let 'vector' win by volume
)
AS
SELECT m.token_count, m.freshness_days, m.has_table, m.heading_depth, m.doc_type, m.language,
       l.route
FROM `{PROJECT_ID}.{DATASET}.chunk_metadata` m
JOIN `{PROJECT_ID}.{DATASET}.route_labels` l USING (chunk_id)
WHERE m.tenant_id = '{TENANT}'
"""
run(ROUTER)

for r in run(f"""
  SELECT ROUND(log_loss, 4) AS log_loss, ROUND(roc_auc, 4) AS roc_auc
  FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET}.retrieval_router`)
"""):
    print(f"  log_loss={r.log_loss}  roc_auc={r.roc_auc}")

# On a small label set these are training-set metrics, exactly as lesson 5.1 warned: AUTO_SPLIT
# holds back an evaluation split only from about 500 rows. Label more before you believe them.

### Predict, and notice what is absent
No TRANSFORM code appears in the prediction query. `ML.PREDICT` applied the bucketising and the casts from the model itself.

> Compare the cost with lesson 3.3, which classified query complexity with a Flash-Lite call before every real call. This router decides from features already computed, so at query time it is a table lookup. The trade: it routes on what it knows about the **chunk**, not the question — which is why 12.6 keeps both.

In [ ]:
# What 12.6 wires into router.py: a lookup, not a model call at query time.
PREDICT = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.chunk_route` AS
SELECT chunk_id,
       predicted_route,
       (SELECT MAX(p.prob) FROM UNNEST(predicted_route_probs) p) AS route_confidence
FROM ML.PREDICT(
  MODEL `{PROJECT_ID}.{DATASET}.retrieval_router`,
  (SELECT chunk_id, token_count, freshness_days, has_table, heading_depth, doc_type, language
   FROM `{PROJECT_ID}.{DATASET}.chunk_metadata`
   WHERE tenant_id = '{TENANT}')
)
"""
run(PREDICT)

for r in run(f"""
  SELECT predicted_route AS route, COUNT(*) AS chunks,
         ROUND(AVG(route_confidence), 3) AS avg_confidence
  FROM `{PROJECT_ID}.{DATASET}.chunk_route`
  GROUP BY route ORDER BY chunks DESC
"""):
    print(f"  {r.route:8} {r.chunks:>4} chunks | avg confidence {r.avg_confidence}")

# Note what did NOT happen: no TRANSFORM code appears here. ML.PREDICT applied the bucketising
# and the casts from the model itself. That is the whole argument for TRANSFORM.

## Cell 9: The gate — block the rebuild, not an inbox
Three rules: `chunk_id` is unique, `token_count` sits in a sane band, and nothing PII-flagged reached the index feed.

> **The product was renamed; the API was not.** Dataplex Universal Catalog became **Knowledge Catalog** on 10 April 2026, but the API, client libraries, `gcloud` surface and IAM roles all still say `dataplex` — which is why every command below does. The same pattern holds for Cloud DLP, now Sensitive Data Protection with the DLP API unchanged.

In [ ]:
# Knowledge Catalog auto data quality. The product was renamed from Dataplex Universal Catalog
# on 2026-04-10; the API, the CLI and the IAM roles still say `dataplex`, which is why every
# command below does too. Verified 2026-09-03.
#
# These three rules are the gate. If they fail, the index does not get rebuilt.
# The spec FILE is the DataQualitySpec itself - `rules` at the top level, optionally with
# samplingPercent and rowFilter. The table it scans is not in this file: it comes from
# --data-source-resource on the command line. Nest the rules under a "dataQualitySpec" key and
# gcloud finds no rules, creates nothing, and the gate below fails for the wrong reason.
DQ_SPEC = {
    "rules": [
        # 1. Uniqueness: one row per chunk. A duplicate means the merge key is wrong, and a
        #    duplicated chunk is a duplicated citation.
        #    NOTE the scan target: index_feed, not chunk_metadata. chunk_metadata is what you
        #    HAVE; index_feed is what you SHIP, and only the second one can hurt a user. Point
        #    the same three rules at chunk_metadata and rule 2 fails, because the corpus really
        #    does contain two chunks below the floor - that is the exercise at the end.
        {"column": "chunk_id", "dimension": "UNIQUENESS", "uniquenessExpectation": {}},

        # 2. Range: a chunk outside this band is a chunking bug, not a document. Below 32 tokens
        #    it carries no answer; above 2,048 it will not survive 4.5's packer. On the feed this
        #    is a REGRESSION TEST: it asserts that the subtraction in Step 6 still happens. The
        #    day someone "simplifies" that WHERE clause, this rule goes red before users do.
        {"column": "token_count", "dimension": "VALIDITY",
         "rangeExpectation": {"minValue": "32", "maxValue": "2048"},
         "threshold": 0.99},

        # 3. The PII assertion, written as custom SQL because it is a claim about the JOIN between
        #    two tables: nothing flagged as PII may be present in the index feed.
        # `column` and `threshold` are row-level-rule fields. A sqlAssertion is a table-level
        # rule: it passes when the statement returns no rows, so it takes neither.
        {"dimension": "VALIDITY",
         "sqlAssertion": {"sqlStatement":
             f"SELECT chunk_id FROM `{PROJECT_ID}.{DATASET}.chunk_metadata` "
             f"WHERE pii_flag AND chunk_id IN "
             f"(SELECT chunk_id FROM `{PROJECT_ID}.{DATASET}.index_feed`)"}},
        # A sqlAssertion passes when its statement returns NO rows. Read it as the claim you are
        # making - "nothing flagged as PII is in the feed" - and the SQL as the attempt to
        # falsify it. That is the opposite of how most people write a check, and it is why this
        # one cannot pass by accident on an empty table.
    ],
}

import json, subprocess
open("dq_spec.json", "w").write(json.dumps(DQ_SPEC, indent=2))
print(json.dumps(DQ_SPEC["rules"], indent=2)[:900])

# Create the scan, or update it if it is already there. `datascans create` returns ALREADY_EXISTS
# on a second run, so the fallback has to be a real `update` call - an `echo` would leave you
# running yesterday's rules while believing you had changed them. The data source is immutable, so
# only the spec file goes to update.
_SRC = (f"//bigquery.googleapis.com/projects/{PROJECT_ID}"
        f"/datasets/{DATASET}/tables/index_feed")

!gcloud dataplex datascans create data-quality documind-chunk-dq \
    --project=$PROJECT_ID --location=us-central1 \
    --data-quality-spec-file=dq_spec.json --data-source-resource="$_SRC" \
 || gcloud dataplex datascans update data-quality documind-chunk-dq \
    --project=$PROJECT_ID --location=us-central1 \
    --data-quality-spec-file=dq_spec.json

!gcloud dataplex datascans run documind-chunk-dq --project=$PROJECT_ID --location=us-central1

### A scan nobody blocks on is a dashboard
Lesson 4.7 gated on whether the **answers** were faithful. This gates on whether the **data** is fit to index at all — earlier, and far cheaper. A chunking regression shows up here as a range violation in seconds; in 4.7 it shows up as a faithfulness score that dropped for reasons you must then investigate. Run both, and run this one first.

In [ ]:
# A scan that nobody blocks on is a dashboard. This is the gate: read the last run and fail loudly.
def dq_gate(scan: str = "documind-chunk-dq", location: str = "us-central1") -> bool:
    out = subprocess.run(
        ["gcloud", "dataplex", "datascans", "describe", scan,
         "--project", PROJECT_ID, "--location", location,
         "--view=FULL", "--format=json"],
        capture_output=True, text=True)
    if out.returncode != 0:
        print("could not read the scan:", out.stderr[:200]); return False
    result = json.loads(out.stdout).get("dataQualityResult", {})
    passed = result.get("passed", False)
    for rule in result.get("rules", []):
        name = rule.get("rule", {}).get("column", "custom")
        ok = rule.get("passed")
        pct = rule.get("passRatio", 0)
        print(f"  {'PASS' if ok else 'FAIL'}  {name:14} pass ratio {pct}")
    print("GATE:", "PASS - safe to rebuild the index" if passed else "FAIL - index rebuild blocked")
    return passed

dq_gate()

# 4.7 gated on whether the ANSWERS were good. This gates on whether the DATA is fit to index at
# all - which is the earlier and cheaper place to catch the same class of failure.

## Cell 10: Snapshot before you overwrite
A table snapshot stores metadata, not a copy of the bytes, so it is cheap. It is the artefact that lets you say **which features produced last week's answers** when someone disputes one. BigQuery time travel reaches back only 7 days; a snapshot is what extends that.

In [ ]:
# Before the nightly rebuild overwrites chunk_metadata, snapshot it. A snapshot is metadata
# only: it stores no copy of the bytes, so it is cheap, and it is the artefact that lets you say
# WHICH features produced last week's answers when someone disputes one.
# Verified 2026-09-03: bq cp --snapshot --no_clobber --expiration=SECONDS src dst.
import datetime as _dt
STAMP = _dt.date.today().strftime("%Y%m%d")   # computed in Python, not by the shell
!bq cp --snapshot --no_clobber --expiration=2592000 \
    $PROJECT_ID:rag_data.chunk_metadata \
    $PROJECT_ID:rag_data.chunk_metadata_snap_{STAMP} || echo "snapshot exists for today"

# 30 days of expiry (2,592,000 seconds) because that is the window in which a disputed answer is
# still worth reconstructing. BigQuery time travel covers only the last 7 days, so a snapshot is
# what extends the reach.

## Cell 11: Turn the corpus into eval candidates
> **These are candidates, not a golden set.** Lesson 4.7 was explicit that a golden row is a contract a human writes. Questions generated from the corpus inherit the generator's blind spots, so a set built only this way measures how well your retriever matches the model that wrote the questions — not how well it serves users. Review, cut most of them, merge the survivors.

The filters matter: nothing PII-flagged, nothing below the confidence cut, nothing outside the token band — because an eval row built on a chunk you would not index tests a system you are not running.

In [ ]:
# make-evalset: turn the corpus into question/answer pairs that lesson 4.7 can score against.
# Two rules make this safe: PII-flagged chunks never enter, and the output is JSONL that a human
# reviews before anything trains on it.
import json

EVALSET = rf"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.evalset_raw` AS
SELECT chunk_id, question, expected_answer
FROM AI.GENERATE_TABLE(
  MODEL `{PROJECT_ID}.{DATASET}.classifier_model`,
  (
    SELECT c.chunk_id,
           CONCAT('Write one question a colleague would ask that this passage answers, and the ',
                  'answer, quoting the passage. Do not invent facts.\n\nPassage:\n',
                  SUBSTR(c.content, 1, 1500)) AS prompt
    FROM `{PROJECT_ID}.{DATASET}.chunk_source` c
    JOIN `{PROJECT_ID}.{DATASET}.chunk_metadata` m USING (chunk_id)
    WHERE m.tenant_id = '{TENANT}'
      AND NOT m.pii_flag                         -- the gate, again
      AND m.token_count BETWEEN 32 AND 2048      -- the same floor the quality rule enforces
      AND m.doc_type_conf >= 0.7                 -- do not build eval rows on guesses
  ),
  STRUCT(
    "question STRING OPTIONS(description = 'One natural question the passage answers'), "
    "expected_answer STRING OPTIONS(description = 'The answer, quoting the passage')" AS output_schema
  )
)
"""
run(EVALSET)

rows = run(f"""
  SELECT chunk_id, question, expected_answer
  FROM `{PROJECT_ID}.{DATASET}.evalset_raw`
  WHERE question IS NOT NULL LIMIT 200
""")
with open("golden_generated.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps({"id": r.chunk_id, "question": r.question,
                            "expected": r.expected_answer,
                            "must_retrieve": [r.chunk_id],
                            "answerable": True, "source": "generated"},
                           ensure_ascii=False) + "\n")
print(f"wrote {len(rows)} rows -> golden_generated.jsonl")

# These are CANDIDATE rows, not a golden set. Lesson 4.7's golden set is written by a human and is
# the contract; this is raw material for it. Generated questions inherit the model's blind spots,
# so a set built only this way measures how well you match the generator, not the user. Review,
# cut, and merge the survivors into deploy/evals/golden.jsonl.

### The same step, as the kit's script
`deploy/evals/make_evalset.py` reads only the chunks the gate allowed, asks Gemini for one pair per chunk, scans every pair with the one PII list, and writes candidates that `run_eval.py` will not read.


In [ ]:
# The kit's version of this step is a script, not a SQL statement: deploy/evals/make_evalset.py reads the chunks the
# quality gate ALLOWED (index_feed joined to chunk_source, so nothing PII-flagged and nothing outside the token band
# becomes an eval row), asks Gemini for one question/answer pair per chunk as a structured call, scans every pair
# with the one PII list (shared/pii.py) and writes golden_generated.jsonl in 4.7's row shape with shape="generated".
# run_eval.py will not read that file, on purpose: these are candidates a person reviews. make make-evalset runs it.
import os, subprocess
KIT = "/content/agentic-ai-weekend-gcp-learners"
if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", "main",
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
src = open(f"{KIT}/deploy/evals/make_evalset.py", encoding="utf-8").read()
print(src.split('"""')[1].strip())          # the docstring: what it reads, what it refuses, where it writes
print()
print("\n".join(l for l in src.splitlines() if l.startswith(("def ", "OUT = ", "MODEL = ")) or "shared.pii" in l))


## What Module 5 built

In [ ]:
# What Module 5 built, end to end.
for r in run(f"""
  SELECT
    (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET}.chunk_metadata`)          AS chunks_featured,
    (SELECT COUNTIF(pii_flag) FROM `{PROJECT_ID}.{DATASET}.chunk_metadata`) AS withheld,
    (SELECT COUNT(DISTINCT doc_type) FROM `{PROJECT_ID}.{DATASET}.chunk_metadata`) AS doc_types,
    (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET}.ingest_events`)           AS events_logged,
    (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET}.evalset_raw`)             AS eval_candidates
"""):
    print(f"  chunks featured : {r.chunks_featured}")
    print(f"  withheld (PII)  : {r.withheld}")
    print(f"  doc types found : {r.doc_types}")
    print(f"  ingest events   : {r.events_logged}")
    print(f"  eval candidates : {r.eval_candidates}")

print("""
5.1 taught CREATE MODEL.            5.5 trains the routing classifier with TRANSFORM.
5.2 taught forecasting.             5.5 gives freshness_days, the feature that decays.
5.3 taught AI.GENERATE.             5.5 runs AI.GENERATE_TABLE over the whole corpus.
5.4 taught the registry and schedules. 5.5 is what the nightly schedule runs.
""")

## ✅ Lesson 5.5 complete — and Module 5 with it

- ✅ Separated features that cost nothing from the one that costs a model call, and scheduled each accordingly
- ✅ Classified a whole corpus in one `AI.GENERATE_TABLE` statement
- ✅ Built a PII gate by subtraction rather than by a filter someone can forget
- ✅ Turned feature rows into `restricts` and `numeric_restricts`, and can say why AND-across-namespaces makes namespace design a security decision
- ✅ Put a sparse vector beside the dense one in the same index
- ✅ Trained a routing classifier whose preprocessing cannot drift from training
- ✅ Wrote a data-quality gate that blocks an index rebuild

**Module 5:** 5.1 CREATE MODEL → 5.2 forecasting and anomalies → 5.3 Gemini inside SQL → 5.4 registry, schedules and what Feature Store is no longer for → 5.5 the features, filters, router and gate.

**Next: Module 6 — Gemini Function Calling.** Everything so far answers from documents; Module 6 lets DocuMind *act*. The `chunk_route` table you just built is read by lesson 12.6 when the router is wired into the API.